# Notebook 03 — Finding a Needle in 2 Billion Rows
## Search Optimization Service (SOS)

**The scenario**: A fraud analyst needs to pull all transactions for one specific account. With 2 billion rows and 50M unique accounts, Snowflake's partition metadata can't efficiently narrow down which partitions contain a given account.

**The fix**: Search Optimization Service (SOS) builds a special index for equality lookups. Think of it like the index at the back of a book — instead of reading every page, you jump directly to what you need.

| Table | SOS Enabled? | Equality Lookup Behavior |
|-------|-------------|------------------------|
| `TRANSACTIONS_NO_SOS` | No | Scans ~42 GB to find one account |
| `TRANSACTIONS_SOS` | Yes (account_id) | Jumps directly to target partitions (~0.7 GB) |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

In [ ]:
-- Verify SOS is active (should show 'ACTIVE' status)
DESCRIBE SEARCH OPTIMIZATION ON TRANSACTIONS_SOS;

### Track SOS Indexing Progress

Before benchmarking, confirm the search access path is fully built. The `search_optimization_progress` column shows percentage complete (100 = fully indexed).

In [ ]:
-- Check SOS indexing progress (must be 100 before benchmarking)
SHOW TABLES LIKE 'TRANSACTIONS_SOS' IN SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

SELECT "name",
       "search_optimization",
       "search_optimization_progress",
       ROUND("search_optimization_bytes" / (1024*1024*1024), 2) AS index_size_gb
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

---
## Account ID Lookup — SOS vs No SOS

"Pull all transactions for account ACC-0000012345."

### Worst Case: No SOS

In [ ]:
-- WORST: No SOS — must scan broadly to find this account
SELECT
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_NO_SOS
WHERE account_id = 'ACC-0000012345';

### Best Case: With SOS

In [ ]:
-- BEST: SOS index narrows the search immediately
SELECT
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_SOS
WHERE account_id = 'ACC-0000012345';

---
## Compare Results

In [ ]:
-- Compare bytes scanned and time
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_SOS%' THEN 'WITH SOS (best)'
        ELSE 'NO SOS (worst)'
    END AS approach,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_WAREHOUSE(
    WAREHOUSE_NAME => 'HOL_XS',
    END_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_text NOT ILIKE '%CLUSTERING_INFORMATION%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Key Takeaway

| Metric | Without SOS | With SOS | Improvement |
|--------|------------|----------|-------------|
| GB Scanned | ~42 GB | ~0.7 GB | **58x less** |
| Time | ~10 sec | ~1.2 sec | **8x faster** |

**SOS vs Clustering — when to use which:**

| Feature | Clustering | Search Optimization (SOS) |
|---------|-----------|---------------------------|
| Best for | Range queries, common filters | Exact-match equality lookups |
| Works on | 1-2 key columns | Multiple columns simultaneously |
| Maintenance | Snowflake auto-reclusters | Snowflake auto-maintains index |
| Cost model | Reclustering credits | Storage + maintenance credits |
| Example | `WHERE state = 'NY'` | `WHERE account_id = '...'` |

**When to ask your data engineer for SOS:**
- You need instant lookups by ID (fraud investigations, customer service)
- The column has millions of unique values
- You always use `=` (not `BETWEEN` or `LIKE`)

**Next** → [Notebook 04 — Warehouse Sizing](./Notebook_04_Warehouse_Sizing.ipynb) (what happens when your query runs out of memory)